# Project 2

## Imports

In [2]:
!pip install spacy torch textworld[gym] tqdm scikit-learn && python -m spacy download en_core_web_sm

  Using cached spacy-3.8.7-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.13-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.2 kB)
  Using cached cymem-2.0.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.5 kB)
  Using cached preshed-3.0.10-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.4 kB)
  Using cached thinc-8.3.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_

In [4]:
import os
import json
import random
import re

import spacy
import torch
import textworld
import textworld.gym

from collections import Counter
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

ModuleNotFoundError: No module named 'textworld'

## Part 1


### Data collection 

In [5]:
def parse_verb_object(s):
    return s.split()[0], ' '.join(s.split()[1:])

def walkthrough_and_record(game_dir, max_games=500):

    triples = []
    seen = 0

    files = list(os.listdir(game_dir))

    # let's make it reproducible
    random.seed(42)
    random.shuffle(files)

    for file in files:

        # jsons have the walkthroughs... if no json, no need to look at this game
        if not file.endswith('.json'): continue

        # cap the total games for speed
        seen += 1
        if seen > max_games: break

        # extract walkthrough from the json meta data
        # walkthrough == winning actions
        x = json.load(open(os.path.join(game_dir, file), 'r'))
        wt = x['extras']['walkthrough']

        # setup the actual game we can interact with to get observations
        ulx = os.path.join(game_dir, file.replace('.json', '.ulx'))
        game_id = textworld.gym.register_game(ulx)
        env = textworld.gym.make(game_id)

        # record observations and track the moves we made at each to win
        obs, *_ = env.reset()

        for move in wt:
            triples.append((obs, *parse_verb_object(move)))
            obs, *_ = env.step(move)

        # don't forget the last move we just made
        triples.append((obs, *parse_verb_object(move)))

    return triples

gdir = 'cog2019_ftwp/games/train'

# triples of obs, verb, obj
print('Extracting data for ML... this could take a while')
ml_data = walkthrough_and_record(gdir)
print('> We extracted', len(ml_data), '(obs, act) pairs.')

Extracting data for ML... this could take a while


FileNotFoundError: [Errno 2] No such file or directory: 'cog2019_ftwp/games/train'

## Part 2